# **章节实践**

通过本章的系统学习，我们已经掌握了如何基于ops-math仓进行Ascend C开发。为了巩固所学知识，现提供以下实践练习：

基于ops-math仓实现Ascend C算子Sigmoid,算子命名为Sigmoid，编写其kernel侧代码、host侧代码，并完成Infershape UT、 Tiling UT、 Kernel UT测试。  
相关算法：  

**sigmoid(x) = 1/(1 + exp(-x))**

算子原型：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > Sigmoid</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并2行（x行+列名行） -->
    <td rowspan="2" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">[8, 2048]</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">x</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">[8, 2048]</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float / int32</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  
  <!-- 算子输出模块 -->
  <tr>
    <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">y</td>
    <td style="padding: 8px;">与输入x形状一致</td>
    <td style="padding: 8px;">float / int32</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

要求：

- 完成Sigmoid算子实现。

- 完成Sigmoid算子UT测试。

- 要支持float类型输入输出。

- 支持输入x的shape修改为(8, 2048)，类型为float32的UT测试。

请开始你的实践，体验完整开发过程。

---

环境准备：

In [ ]:
!mkdir -p Sources/06.05

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

ops-math仓创建新算子工程时依赖gawk，执行以下命令安装gawk

In [ ]:
%%bash
set -e

GAWK_VERSION="5.3.0"
INSTALL_DIR="$HOME/.local"
RC_FILE="$HOME/.bashrc"  # 固定使用.bashrc，不存在则创建
PATH_LINE="export PATH=$INSTALL_DIR/bin:\$PATH"
# 确保.bashrc存在，且只添加一次PATH
touch "$RC_FILE" 
source "$RC_FILE"
if command -v gawk &> /dev/null; then
    echo "✅ gawk 已存在，版本：$(gawk --version | head -n1)"
    echo "✅ 跳过编译安装"
    exit 0
fi

[ ! -f "gawk-$GAWK_VERSION.tar.gz" ] && wget https://ftp.gnu.org/gnu/gawk/gawk-$GAWK_VERSION.tar.gz

tar -zxf gawk-$GAWK_VERSION.tar.gz --overwrite 
cd gawk-$GAWK_VERSION
./configure --prefix=$INSTALL_DIR
make -j$(nproc)
make install

grep -q "$INSTALL_DIR/bin" "$RC_FILE" || echo "$PATH_LINE" >> "$RC_FILE"
source "$RC_FILE"

echo "gawk 安装完成，路径：$(which gawk)"

In [ ]:
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

ops-math仓准备

In [ ]:
%%bash
# 清除可能存在的ops-math，以确保下载正常
rm -rf Sources/06.05/ops-math
# 下载ops-math开源仓
cd Sources/06.05;git clone https://gitcode.com/cann/ops-math.git;cd ..

### **算子工程创建**

In [ ]:
%cd Sources/06.05/ops-math 
!bash build.sh --genop=experimental/math/sigmoid

### **Tiling实现文件修改**  
根据要求完成tiling实现

In [ ]:
%%writefile ./experimental/math/sigmoid/op_host/sigmoid_tiling.cpp
#include "log/log.h"
#include "util/math_util.h"
#include "register/op_impl_registry.h"
#include <graph/utils/type_utils.h>
#include "tiling/platform/platform_ascendc.h"
#include "../op_kernel/sigmoid_tiling_data.h"
#include "../op_kernel/sigmoid_tiling_key.h"

namespace optiling {

struct SigmoidCompileInfo {};

// tiling 分发入口
static ge::graphStatus SigmoidTilingFunc(gert::TilingContext* context)
{
    // 补充tiling实现
}

static ge::graphStatus TilingParseForSigmoid([[maybe_unused]] gert::TilingParseContext* context)
{   
    // AscendC 算子可以直接返回SUCCESS提升性能，硬件信息可在运行时获取
    return ge::GRAPH_SUCCESS;
}

// tiling注册入口.
IMPL_OP_OPTILING(Sigmoid).Tiling(SigmoidTilingFunc).TilingParse<SigmoidCompileInfo>(TilingParseForSigmoid);
} 

### **原型定义**
根据要求完成原型定义修改

In [ ]:
%%writefile ./experimental/math/sigmoid/op_host/sigmoid_def.cpp
#include "register/op_def_registry.h"

namespace ops {
class Sigmoid : public OpDef {
public:
    explicit Sigmoid(const char* name) : OpDef(name)
    {
        // 输入参数说明
        this->Input("x1")                                       // 输入x1定义
            .ParamType(REQUIRED)                                // 必选输入
            .DataType({ge::DT_FLOAT, ge::DT_INT32})             // 支持数据类型
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})             // 支持format格式
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND}) // 未确定大小shape对应format格式
            .AutoContiguous();                                  // 内存自动连续化
        
        /* ...此处补充其他输入输出参数说明 */

        // 输出参数说明
        this->Output("y") // 输出y定义
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT, ge::DT_INT32})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND})
            .AutoContiguous();

        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(Sigmoid); // 添加算子信息库
}

### **InferShape实现**
根据要求完成InferShape实现修改

In [ ]:
%%writefile ./experimental/math/sigmoid/op_host/sigmoid_infershape.cpp
#include "register/op_impl_registry.h"
#include "log/log.h"

using namespace ge;

namespace ops {

static ge::graphStatus InferShapeSigmoid(gert::InferShapeContext* context)
{
    OP_LOGD(context->GetNodeName(), "Begin to do InferShapeSigmoid");

    // get input shapes
    const gert::Shape* xShape = context->GetInputShape(IDX_0);
    OP_CHECK_NULL_WITH_CONTEXT(context, xShape);

    // get output shapes
    gert::Shape* yShape = context->GetOutputShape(IDX_0);
    OP_CHECK_NULL_WITH_CONTEXT(context, yShape);

    // 填充输出shape大小
    auto xShapeSize = xShape->GetDimNum();
    yShape->SetDimNum(xShapeSize);
    for (size_t i = 0; i < xShapeSize; i++) {
        int64_t dim = xShape->GetDim(i);
        yShape->SetDim(i, dim);
    }

    OP_LOGD(context->GetNodeName(), "End to do InferShapeSigmoid");
    return GRAPH_SUCCESS;
}

IMPL_OP_INFERSHAPE(Sigmoid).InferShape(InferShapeSigmoid);
}

### **Tiling结构体定义**
根据要求完成Tiling结构体定义修改

In [ ]:
%%writefile ./experimental/math/sigmoid/op_kernel/sigmoid_tiling_data.h
#ifndef __SIGMOID_TILLING_DATA_H__
#define __SIGMOID_TILLING_DATA_H__

struct SigmoidTilingData {
    int64_t totalLength;
    int64_t tileNum;
    // 扩展其他tilling参数
};
#endif

### **TilingKey模板定义**
根据要求完成TilingKey模板定义修改

In [ ]:
%%writefile ./experimental/math/sigmoid/op_kernel/sigmoid_tiling_key.h
#ifndef __SIGMOID_TILING_KEY_H__
#define __SIGMOID_TILING_KEY_H__

#include "ascendc/host_api/tiling/template_argument.h"

/* Mode场景定义 */
#define ELEMENTWISE_TPL_SCH_MODE_0 0
#define ELEMENTWISE_TPL_SCH_MODE_1 1
/* 继续定义其他Mode场景... */

/* 模板参数 */
ASCENDC_TPL_ARGS_DECL(
    Sigmoid,
    ASCENDC_TPL_UINT_DECL(schMode, 1, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0, ELEMENTWISE_TPL_SCH_MODE_1));

/* 模板参数组合 */
ASCENDC_TPL_SEL(ASCENDC_TPL_ARGS_SEL(
    ASCENDC_TPL_UINT_SEL(schMode, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0, ELEMENTWISE_TPL_SCH_MODE_1)));

#endif

### **算法实现**
根据要求完成算法实现修改

In [ ]:
%%writefile ./experimental/math/sigmoid/op_kernel/sigmoid.h
#ifndef __SIGMOID_H__
#define __SIGMOID_H__

#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "sigmoid_tiling_data.h"
#include "sigmoid_tiling_key.h"

namespace NsSigmoid {

using namespace AscendC;

constexpr int32_t BUFFER_NUM = 2;
constexpr int32_t QUEUE_DEPTH = 2;

template <typename T>
class Sigmoid {
public:
    __aicore__ inline Sigmoid(){};

    __aicore__ inline void Init(/*参数列表*/);
    __aicore__ inline void Process(/*参数列表*/);

private:
    __aicore__ inline void CopyIn(/*参数列表*/);
    __aicore__ inline void CopyOut(/*参数列表*/);
    __aicore__ inline void Compute(/*参数列表*/);

private:
    TPipe pipe;
    TQue<QuePosition::VECIN, QUEUE_DEPTH> XXX;
    TQue<QuePosition::VECOUT, QUEUE_DEPTH> YYY;
};

template <typename T>
__aicore__ inline void Sigmoid<T>::Init(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sigmoid<T>::CopyIn(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sigmoid<T>::CopyOut(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sigmoid<T>::Compute(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sigmoid<T>::Process()
{
    for (int32_t i = 0; i < /*循环次数*/; i++) {
        CopyIn(/*参数列表*/);
        Compute(/*参数列表*/);
        CopyOut(/*参数列表*/);
    }
}

} // namespace NsSigmoid
#endif // SIGMOID_H

### **核函数实现**
根据要求完成核函数实现修改

In [ ]:
%%writefile ./experimental/math/sigmoid/op_kernel/sigmoid.cpp
#include "sigmoid.h"

enum class SigmoidTilingKey : uint32_t
{
    TILING_KEY_EXAMPLE_FLOAT = 0,
    TILING_KEY_EXAMPLE_INT32 = 1,
};

template <uint32_t schMode>
__global__ __aicore__ void sigmoid(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(SigmoidTilingData);
    GET_TILING_DATA_WITH_STRUCT(SigmoidTilingData, tilingData, tiling);

    // 场景1
    if constexpr (schMode == static_cast<uint32_t>(SigmoidTilingKey::TILING_KEY_EXAMPLE_FLOAT)) {
        NsSigmoid::Sigmoid<float> op; // 算子kernel实例获取
        op.Init(x, y, z, &tilingData);      // 算子kernel实例初始化
        op.Process();                       // 算子kernel实例执行
    }

    // 场景2
    if constexpr (schMode == static_cast<uint32_t>(SigmoidTilingKey::TILING_KEY_EXAMPLE_INT32)) {
        NsSigmoid::Sigmoid<int32_t> op; // 算子kernel实例获取
        op.Init(x, y, z, &tilingData);        // 算子kernel实例初始化
        op.Process();                         // 算子kernel实例执行
    }
}

### **编写UT测试用例**
完成Infershape UT、Tiling UT、Kernel UT用例编写

In [ ]:
# 创建UT存放文件夹
!mkdir -p experimental/math/sigmoid/tests/ut/op_host
!mkdir -p experimental/math/sigmoid/tests/ut/op_kernel
!mkdir -p experimental/math/sigmoid/tests/ut/op_kernel/sigmoid_data
# 创建空UT文件
!touch experimental/math/sigmoid/tests/ut/op_host/test_sigmoid_infershape.cpp
!touch experimental/math/sigmoid/tests/ut/op_host/test_sigmoid_tiling.cpp
!touch experimental/math/sigmoid/tests/ut/op_kernel/test_sigmoid.cpp
!touch experimental/math/sigmoid/tests/ut/op_kernel/sigmoid_data/gen_data.py
!touch experimental/math/sigmoid/tests/ut/op_kernel/sigmoid_data/compare.py

编写Infershape UT

In [ ]:
%%writefile experimental/math/sigmoid/tests/ut/op_host/test_sigmoid_infershape.cpp   
#include <gtest/gtest.h>
#include <iostream>
#include "infershape_context_faker.h"
#include "infershape_case_executor.h"

class SigmoidInfershapeTest : public testing::Test
{
protected:
    static void SetUpTestCase()
    {
        std::cout << "SigmoidInfershapeTest SetUp" << std::endl;
    }

    static void TearDownTestCase()
    {
        std::cout << "SigmoidInfershapeTest TearDown" << std::endl;
    }
};

TEST_F(SigmoidInfershapeTest, sigmoid_infershape_test1)
{

}

编写Tiling UT

In [ ]:
%%writefile experimental/math/sigmoid/tests/ut/op_host/test_sigmoid_tiling.cpp
#include <iostream>
#include <gtest/gtest.h>
#include "tiling_context_faker.h"
#include "tiling_case_executor.h"

class SigmoidTilingTest : public testing::Test {
protected:
    static void SetUpTestCase()
    {
        std::cout << "SigmoidTilingTest SetUp" << std::endl;
    }

    static void TearDownTestCase()
    {
        std::cout << "SigmoidTilingTest TearDown" << std::endl;
    }
};

TEST_F(SigmoidTilingTest, sigmoid_tiling_test_1)
{

}

编写Kernel UT

In [ ]:
%%writefile experimental/math/sigmoid/tests/ut/op_kernel/test_sigmoid.cpp
#include "../../../op_kernel/sigmoid.cpp"  
#include <array>
#include <vector>
#include <iostream>
#include <string>
#include <cstdint>
#include <cstdlib>  
#include "gtest/gtest.h"
#include "tikicpulib.h"
#include "data_utils.h"

using namespace std;

class SigmoidTest : public testing::Test {
protected:
    static void SetUpTestCase()
    {
        cout << "SigmoidTest SetUp\n" << endl;
        const string cmd = "cp -rf " + dataPath + " ./";
        system(cmd.c_str());
        system("chmod -R 755 ./sigmoid_data/");
    }

    static void TearDownTestCase()
    {
        cout << "SigmoidTest TearDown\n" << endl;
    }

private:
    const static std::string rootPath;
    const static std::string dataPath;
};

const std::string SigmoidTest::rootPath = "../../../../";  
const std::string SigmoidTest::dataPath = rootPath + "experimental/math/sigmoid/tests/ut/op_kernel/sigmoid_data";

TEST_F(SigmoidTest, test_case_0)
{

}

编写数据生成脚本

In [ ]:
%%writefile experimental/math/sigmoid/tests/ut/op_kernel/sigmoid_data/gen_data.py
import numpy as np
import os

def gen_golden_data_simple():

    input_x.tofile("./input.bin")
    golden.tofile("./golden.bin")


if __name__ == "__main__":
    # 清理bin文件
    os.system("rm -rf *.bin")
    gen_golden_data_simple()

编写对比脚本

In [ ]:
%%writefile experimental/math/sigmoid/tests/ut/op_kernel/sigmoid_data/compare_data.py
import sys
import numpy as np
import glob
import os

curr_dir = os.path.dirname(os.path.realpath(__file__))

def compare_data(golden_file_lists, output_file_lists, d_type):
    np_dtype = d_type
    data_same = True
    for gold, out in zip(golden_file_lists, output_file_lists):
        tmp_out = np.fromfile(out, np_dtype)
        tmp_gold = np.fromfile(gold, np_dtype)
        diff_res = np.isclose(tmp_out, tmp_gold, 1e-4, 1e-4, True)
        diff_idx = np.where(diff_res != True)[0]
        if len(diff_idx) == 0:
            print("PASSED!")
        else:
            print("FAILED!")
            for idx in diff_idx[:5]:
                print(f"index: {idx}, output: {tmp_out[idx]}, golden: {tmp_gold[idx]}")
            data_same = False
    return data_same

def get_file_lists(dtype):
    golden_file_lists = sorted(glob.glob(curr_dir + "/*golden*.bin"))
    output_file_lists = sorted(glob.glob(curr_dir + "/*output*.bin"))
    return golden_file_lists, output_file_lists

def process(d_type):
    golden_file_lists, output_file_lists = get_file_lists(d_type)
    result = compare_data(golden_file_lists, output_file_lists, d_type)
    print("compare result:", result)
    return result

if __name__ == '__main__':
    ret = process(np.float32)
    exit(0 if ret else 1)

编写Kernel UT编译脚本

In [ ]:
%%writefile experimental/math/sigmoid/tests/ut/op_kernel/CMakeLists.txt
if (UT_TEST_ALL OR OP_KERNEL_UT)
    # 需要将Tiling依赖的文件添加到CMakeLists.txt中
    # set(elewise_common_tiling_files
    #         ${CANN_ROOT}/ops/built-in/op_tiling/runtime/elewise_tiling.cc
    #         )
    # 算子自己的tiling文件路径
    set(sigmoid_tiling_files
        ${CMAKE_CURRENT_SOURCE_DIR}/../../../op_host/sigmoid_tiling.cpp
        )
    # 使用AddOpTestCase
    # param1：算子名称，以kernel方式命名
    # param2：soc版本，多个以分号分隔，例如："ascend950PR_99;AscendB1"
    # param3：自定义编译选项，一般填写测试的一种典型数据类型组合，不需要则传入空字符串，例如："-DDTYPE_X=float"，多个使用空格分隔，例如："-DDTYPE_X=float -DDTYPE_Y=float"
    # param4：该算子依赖的所有tiling源码文件
    AddOpTestCase(sigmoid "ascend910b" "" "${sigmoid_tiling_files}")
endif()

### **UT测试**
执行以下代码完成UT测试

In [ ]:
%%bash
# 清除UT临时文件
rm -rf build build_out
# 测试InferShape UT 、Tiling UT
bash build.sh -u --ophost --ops=sigmoid --experimental



预期输出：
```shell
[----------] 1 test from SigmoidTilingTest
SigmoidTilingTest SetUp
[ RUN      ] SigmoidTilingTest.sigmoid_tiling_test_1
[INFO] GE(10248,math_op_host_ut):2026-03-17-17:10:32.195.193 [op_context_builder_impl.cc:103]10248 CreateComputeNodeInfoImpl:Node Sigmoid, compute_node_info attr_size 48, outputs_ins_info_size:48, offset:608, total_size:712.
[       OK ] SigmoidTilingTest.sigmoid_tiling_test_1 (1 ms)
SigmoidTilingTest TearDown
[----------] 1 test from SigmoidTilingTest (1 ms total)

[----------] 1 test from SigmoidInfershapeTest
SigmoidInfershapeTest SetUp
[ RUN      ] SigmoidInfershapeTest.sigmoid_infershape_test1
[INFO] GE(10248,math_op_host_ut):2026-03-17-17:10:32.196.147 [op_context_builder_impl.cc:103]10248 CreateComputeNodeInfoImpl:Node Sigmoid, compute_node_info attr_size 48, outputs_ins_info_size:48, offset:608, total_size:712.
[       OK ] SigmoidInfershapeTest.sigmoid_infershape_test1 (0 ms)
SigmoidInfershapeTest TearDown
[----------] 1 test from SigmoidInfershapeTest (0 ms total)

[----------] Global test environment tear-down
Global Environment TearDown
[==========] 2 tests from 2 test suites ran. (16 ms total)
[  PASSED  ] 2 tests.
```  

执行Kernel UT测试

In [ ]:
%%bash
# 清除UT临时文件
rm -rf build build_out
# 测试Kernel UT
bash build.sh -u --opkernel --ops=sigmoid --experimental

预期输出：
```shell
[       OK ] SigmoidTest.test_case_0 (984 ms)
SigmoidTest TearDown

[----------] 1 test from SigmoidTest (984 ms total)

[----------] Global test environment tear-down
Global Environment TearDown
[==========] 1 test from 1 test suite ran. (1034 ms total)
[  PASSED  ] 1 test.
[100%] Built target math_op_kernel_ut_ascend910B1
[100%] Built target math_op_kernel_ut

```

执行以下代码查看答案：

Tiling实现

In [ ]:
%cd -
!cat ./answer/06.05_answer/op_host/sigmoid_tiling.cpp

原型定义

In [ ]:
!cat ./answer/06.05_answer/op_host/sigmoid_def.cpp

Infershape实现

In [ ]:
!cat ./answer/06.05_answer/op_host/sigmoid_infershape.cpp

Tiling结构体定义

In [ ]:
!cat ./answer/06.05_answer/op_kernel/sigmoid_tiling_data.h

TilingKey模板定义

In [ ]:
!cat ./answer/06.05_answer/op_kernel/sigmoid_tiling_key.h

核函数定义

In [ ]:
!cat ./answer/06.05_answer/op_kernel/sigmoid.cpp

算法实现

In [ ]:
!cat ./answer/06.05_answer/op_kernel/sigmoid.h

Infershape UT实现

In [ ]:
!cat ./answer/06.05_answer/tests/ut/op_host/test_sigmoid_infershape.cpp

Tiling UT实现

In [ ]:
!cat ./answer/06.05_answer/tests/ut/op_host/test_sigmoid_tiling.cpp

Kernel UT实现

In [ ]:
!cat ./answer/06.05_answer/tests/ut/op_kernel/test_sigmoid.cpp

数据生成脚本

In [ ]:
!cat ./answer/06.05_answer/tests/ut/op_kernel/sigmoid_data/gen_data.py